# GFS download on Colab

Use a **CPU** runtime (not a GPU) to save compute units. Nothing here needs a GPU.

Before running: upload `balloon_code.zip` (from `dist/` in the project) into the shared `balloon-forecaster` Drive folder.

Run the cells in order. Cell 1 is the speed test; paste its output back to Claude.

## 1. Speed test to AWS (same test that was slow on the laptop)

In [1]:
import time, requests
from concurrent.futures import ThreadPoolExecutor

URL = 'https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs.20260101/12/atmos/gfs.t12z.pgrb2.0p50.f024'

def fetch(start, nbytes):
    r = requests.get(URL, headers={'Range': f'bytes={start}-{start + nbytes - 1}'}, timeout=120)
    return len(r.content)

t = time.time(); n = fetch(0, 30_000_000); dt = time.time() - t
print(f'1 connection : {n/1e6:.0f} MB in {dt:.1f}s = {n/1e6/dt:.1f} MB/s')

t = time.time()
with ThreadPoolExecutor(8) as ex:
    total = sum(ex.map(lambda i: fetch(40_000_000 + i * 5_000_000, 5_000_000), range(8)))
dt = time.time() - t
print(f'8 connections: {total/1e6:.0f} MB in {dt:.1f}s = {total/1e6/dt:.1f} MB/s total')

1 connection : 30 MB in 1.1s = 27.5 MB/s
8 connections: 40 MB in 0.5s = 73.3 MB/s total


## 2. Mount Drive, unpack the code, install packages

In [5]:
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/balloon-forecaster'
CODE_DIR = '/content/code'
assert os.path.isdir(DATA_ROOT), f'{DATA_ROOT} not found: add the shared folder shortcut to My Drive'

bundle = f'{DATA_ROOT}/balloon_code.zip'
assert os.path.exists(bundle), 'Upload balloon_code.zip into the balloon-forecaster Drive folder first'
zipfile.ZipFile(bundle).extractall(CODE_DIR)
print('code unpacked:', sorted(os.listdir(CODE_DIR)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
code unpacked: ['configs', 'scripts', 'src']


In [6]:
# GRIB reader. ecmwflibs bundles the ecCodes library; apt is a fallback.
!pip -q install netCDF4 cfgrib ecmwflibs

import subprocess, sys
check = subprocess.run([sys.executable, '-c', 'import cfgrib, eccodes; print("cfgrib", cfgrib.__version__, "ok")'],
                       capture_output=True, text=True)
print(check.stdout or check.stderr[-800:])

cfgrib 0.9.15.1 ok



If the check above failed, run this cell, then rerun the check:
```
!apt-get -qq install -y libeccodes-dev && pip -q install eccodes
```

## 3. Pilot download: September 2026 (30 days, all 21 forecast steps)
Measured on Colab: about 20 s per day, so this should take about 10 minutes. It is safe to stop and rerun: finished days are skipped. To change the range, edit START and END below.

In [ ]:
START, END, EVERY = '2026-09-01', '2026-09-30', 1   # pilot month
WORKERS = 4   # forecast steps downloaded at once (each uses 8 connections)

!cd {CODE_DIR} && BALLOON_DATA_DIR={DATA_ROOT} python -u -m scripts.fetch_gfs --start {START} --end {END} --every {EVERY} --workers {WORKERS}

30 days, cycle 12Z, 0.5 degrees, steps 0-120 h


## 4. Verify what landed on Drive

In [ ]:
import glob
import xarray as xr

# Only current-format (v2) files. Open each one fully, which also catches any file damaged
# by a crash at shutdown.
files = sorted(glob.glob(f'{DATA_ROOT}/processed/gfs/gfs_0p50_v2_*.nc'))
bad = []
for f in files:
    try:
        with xr.open_dataset(f) as ds:
            ds.load()
            assert ds.sizes['lead_h'] == 21 and ds.sizes['isobaricInhPa'] == 21
    except Exception as e:
        bad.append((os.path.basename(f), repr(e)))
print(len(files), 'v2 files,', round(sum(os.path.getsize(f) for f in files) / 1e6, 1), 'MB,', len(bad), 'unreadable')
for name, err in bad: print('  BAD', name, err)
for f in files[:5]: print(' ', os.path.basename(f))

## 5. Full history: 2021-03-22 to 2026-10-18 (about 2,000 days)
Before running, **re-upload the rebuilt `balloon_code.zip`** to Drive and rerun cell 2 (it now skips days NOAA does not have and refuses start dates before 2021-03-22).

- Expect roughly 5 to 6 hours with 2 processes (about 20 s per day, split across the CPU cores). Keep the laptop awake and this window connected.
- If it is interrupted for any reason, just rerun the cell below: finished days are skipped.
- A nonzero exit code at the end can be the harmless shutdown crash seen before. The check in the next cell is what counts.
- **Note your compute units before starting and after finishing**, so we know what a full run costs.

In [7]:
import os, subprocess, sys, time, glob
from datetime import date, timedelta

START, END = '2021-03-22', '2026-10-18'   # GFS v16 era onward (see CLAUDE.md for why)
WORKERS = 4                               # forecast steps downloaded at once, per process
N_PROC = min(os.cpu_count() or 2, 4)      # one downloader process per CPU core

# Process i handles days START+i, START+i+N, ... so the processes never overlap. Separate
# processes are fine for the GRIB library; only threads are not.
procs = []
for i in range(N_PROC):
    start_i = (date.fromisoformat(START) + timedelta(days=i)).isoformat()
    log = open(f'/content/fetch_{i}.log', 'w')
    procs.append(subprocess.Popen(
        [sys.executable, '-u', '-m', 'scripts.fetch_gfs', '--start', start_i, '--end', END,
         '--every', str(N_PROC), '--workers', str(WORKERS)],
        cwd=CODE_DIR, env={**os.environ, 'BALLOON_DATA_DIR': DATA_ROOT},
        stdout=log, stderr=subprocess.STDOUT))

total = (date.fromisoformat(END) - date.fromisoformat(START)).days + 1
print(f'{N_PROC} processes, about {total} days to fetch', flush=True)
t0 = time.time()
try:
    while any(p.poll() is None for p in procs):
        time.sleep(120)
        done = len(glob.glob(f'{DATA_ROOT}/processed/gfs/gfs_0p50_v2_*.nc'))
        print(f'{(time.time() - t0) / 60:6.0f} min | {done} files on Drive (goal about {total})', flush=True)
except KeyboardInterrupt:
    for p in procs: p.terminate()
    print('Stopped. Rerun this cell to resume: finished days are skipped.')
for i, p in enumerate(procs):
    print(f'--- process {i}: exit code {p.returncode}. Last lines of its log:')
    print(''.join(open(f'/content/fetch_{i}.log').readlines()[-8:]))

2 processes, about 2037 days to fetch
     2 min | 37 files on Drive (goal about 2037)
     4 min | 43 files on Drive (goal about 2037)
     6 min | 50 files on Drive (goal about 2037)
     8 min | 59 files on Drive (goal about 2037)
    10 min | 67 files on Drive (goal about 2037)
    12 min | 75 files on Drive (goal about 2037)
    14 min | 84 files on Drive (goal about 2037)
    16 min | 93 files on Drive (goal about 2037)
    18 min | 101 files on Drive (goal about 2037)
    20 min | 109 files on Drive (goal about 2037)
    22 min | 117 files on Drive (goal about 2037)
    24 min | 126 files on Drive (goal about 2037)
    26 min | 134 files on Drive (goal about 2037)
    28 min | 143 files on Drive (goal about 2037)
    30 min | 150 files on Drive (goal about 2037)
    32 min | 159 files on Drive (goal about 2037)
    34 min | 168 files on Drive (goal about 2037)
    36 min | 176 files on Drive (goal about 2037)
    38 min | 185 files on Drive (goal about 2037)
    40 min | 194 fil

In [8]:
import glob
from datetime import date, timedelta
import xarray as xr

# Open every file fully (also catches damage from a crash at shutdown) and list any
# days in START..END that have no file.
files = sorted(glob.glob(f'{DATA_ROOT}/processed/gfs/gfs_0p50_v2_*.nc'))
bad = []
for f in files:
    try:
        with xr.open_dataset(f) as ds:
            ds.load()
            assert ds.sizes['lead_h'] == 21 and ds.sizes['isobaricInhPa'] == 21
    except Exception as e:
        bad.append((os.path.basename(f), repr(e)))
have = {os.path.basename(f).split('_')[3] for f in files}      # YYYYMMDD
d0, d1 = date.fromisoformat(START), date.fromisoformat(END)
wanted = [d0 + timedelta(days=i) for i in range((d1 - d0).days + 1)]
missing = [d.isoformat() for d in wanted if d.strftime('%Y%m%d') not in have]
print(len(files), 'v2 files,', round(sum(os.path.getsize(f) for f in files) / 1e6, 1), 'MB,', len(bad), 'unreadable')
print(f'{START} to {END}: {len(wanted) - len(missing)} of {len(wanted)} days present, {len(missing)} missing')
for name, err in bad: print('  BAD', name, err)
print('missing days:', missing[:30], '...' if len(missing) > 30 else '')

2037 v2 files, 4506.1 MB, 0 unreadable
2021-03-22 to 2026-10-18: 2037 of 2037 days present, 0 missing
missing days: [] 


In [ ]:
# LAST CELL of a download session. Colab writes to Drive through a cache: the newest files
# can still be sitting in that cache (visible to this runtime, but not yet on Drive's
# servers). Flushing forces them to upload. This unmounts Drive, so run it last, and only
# after the download processes have finished.
from google.colab import drive
drive.flush_and_unmount()
print('Drive flushed and unmounted. Remount (cell 2) before using Drive again.')